# Cyanite model outputs (tagging)

Fetch Cyanite's per-track audio analysis (the tags) from the **model outputs** endpoint.
This is the core of the *"why this track?"* explainability: every indexed track has rich,
human-readable outputs (genre, mood, instruments, character, BPM, energy, era, and more).

- Endpoint: `GET {BASE_URL}/library-tracks/{track_id}/models?model=...`
- See [../guides/quick_start.md](../guides/quick_start.md) and [../guides/model_outputs.md](../guides/model_outputs.md).
- Tag vocabularies: [../guides/tag_vocabularies.md](../guides/tag_vocabularies.md).

Search (free-text and similarity) is scaffolded at the bottom to fill in later.

## Setup

```bash
pip install requests python-dotenv
```

Put your key in a `.env` file (it is git-ignored, never commit it):

```
CYANITE_API_KEY=your_key_here
```

In [ ]:
import os
import json
import requests
from dotenv import load_dotenv

load_dotenv()  # reads CYANITE_API_KEY from .env

API_KEY = os.environ["CYANITE_API_KEY"]
BASE_URL = "https://rest-api.cyanite.ai/v1"

session = requests.Session()
session.headers.update({"x-api-key": API_KEY})
print("ready")

## The model outputs endpoint

You pass one or more `model` query parameters to select which outputs to return. The full
list of model versions is in the model outputs guide; the ones below are the most useful for
taste / recommendation / explainability.

In [ ]:
# Commonly useful models for this challenge (full list: guides/model_outputs.md)
DEFAULT_MODELS = [
    "MainGenreV2", "SubgenreV2", "FreeGenreV3",
    "MoodSimpleV2", "MoodAdvancedV2",
    "InstrumentsV2", "CharacterV2", "MovementV2",
    "BpmV2", "TempoV1", "KeyV2", "TimeSignatureV2",
    "ValenceArousalV2", "MusicalEraV2", "MusicForV1",
    "VocalsV2", "AutoDescriptionV2", "RepresentativeSegmentV2",
]


def get_model_outputs(track_id, models=None):
    """Fetch selected model outputs for a Cyanite track id.

    Args:
        track_id: a Cyanite library-track id (e.g. 'libtr_...').
        models: list of model version names; defaults to DEFAULT_MODELS.
    Returns:
        Parsed JSON of the model outputs.
    """
    models = models or DEFAULT_MODELS
    params = [("model", m) for m in models]
    resp = session.get(f"{BASE_URL}/library-tracks/{track_id}/models", params=params, timeout=60)
    resp.raise_for_status()
    return resp.json()

In [ ]:
# Replace with a real Cyanite track id. For challenge tracks, resolve a Jamendo id to a
# Cyanite id first (see 'Resolving a track' below).
TRACK_ID = "libtr_REPLACE_ME"

outputs = get_model_outputs(TRACK_ID, ["MainGenreV2", "MoodSimpleV2", "InstrumentsV2", "BpmV2", "AutoDescriptionV2"])
print(json.dumps(outputs, indent=2)[:2500])

## Summarize the tags

A small helper to pull the human-readable bits out of the response for explanations. The
exact JSON shape may differ slightly from your first real response; adjust the accessors
once you see it (the model outputs guide documents each model's fields).

In [ ]:
def iter_model_outputs(outputs):
    """Yield individual model-output dicts regardless of envelope shape."""
    if isinstance(outputs, dict):
        # common envelopes: {"data": [...]}, {"models": [...]}, or {model: {...}}
        for key in ("data", "models", "results"):
            if isinstance(outputs.get(key), list):
                yield from outputs[key]
                return
        for v in outputs.values():
            if isinstance(v, dict) and "version" in v:
                yield v
        return
    if isinstance(outputs, list):
        yield from outputs


def summarize(outputs):
    """Compact human-readable summary: dominant tags / values per model."""
    summary = {}
    for mo in iter_model_outputs(outputs):
        version = mo.get("version", "?")
        if "tags" in mo and mo["tags"] is not None:
            summary[version] = mo["tags"]
        elif "tag" in mo:
            summary[version] = mo["tag"]
        elif "description" in mo:
            summary[version] = mo["description"]
    return summary


summarize(outputs)

## Resolving a track: Jamendo id -> Cyanite id

The data pack (`data/users.csv`, `data/tracks.csv`) uses **Jamendo** numeric ids, while the
API is keyed on **Cyanite** track ids. A lookup will be provided to map between them.

**TODO:** fill this in once the lookup is finalized (a mapping file shipped in `data/`, or an
endpoint). Then `get_model_outputs(resolve_track_id(jamendo_id))` works end to end.

In [ ]:
def resolve_track_id(jamendo_id):
    """Map a Jamendo track id (from the data pack) to a Cyanite track id.

    TODO: implement once the lookup is provided. Likely one of:
      - a mapping file: read data/id_map.csv (jamendo_track_id -> cyanite_track_id)
      - or an API/search lookup by external id.
    """
    raise NotImplementedError("Jamendo->Cyanite id lookup not wired up yet (pending final shape).")

## Free-text search  (to add)

Natural language -> ranked track ids (+ scores), optionally with tag filters. Endpoint TBD
(Search 2.0). The pattern: search to get track ids, then `get_model_outputs` on the ones you
show, and explain via their tags.

In [ ]:
def free_text_search(query, limit=20, filters=None):
    """Free-text / prompt search -> ranked track ids with scores.

    TODO: wire up the Search 2.0 free-text endpoint (path, request body, response shape).
    Expected to return something like: [{"track_id": ..., "score": ...}, ...].
    """
    raise NotImplementedError("Free-text search endpoint TBD.")

## Similarity search  (to add)

Given a seed track id (or a few), return acoustically similar track ids (+ scores). Combine
with `get_model_outputs` to re-rank for tag coherence and to explain each pick.

In [ ]:
def similarity_search(track_id, limit=20, filters=None):
    """Similar-by-id -> ranked track ids with scores.

    TODO: wire up the Search 2.0 similarity endpoint (path, request body, response shape).
    """
    raise NotImplementedError("Similarity search endpoint TBD.")